# Notebook 6 — V-JEPA2 → DinoTXT Integration (No Projection Layer)

This notebook feeds **frozen V-JEPA2 encoder outputs directly into DinoTXT's text-alignment layer**,
without any intermediate projection. This keeps DinoTXT completely unmodified and is the clean
methodological alternative to the projection approach in NB5.

**Key design decisions:**
- V-JEPA2 ViT-L `last_hidden_state` is 1024-dim — same as DINOv2 ViT-L, so DinoTXT receives
  the same dimensionality it was designed for
- Video embedding = **mean pool across all tubelet tokens** → 1024-dim. V-JEPA2 has no
  dedicated CLS token (all tokens are spatiotemporal patch tokens), so mean pooling is the
  architecturally honest choice. It also matches the temporal pooling strategy used for
  DINOv2/v3 in NB4, making the backbone comparison clean.
- V-JEPA2 backbone is fully **frozen** — only DinoTXT's text head trains (same as NB4)
- `skip_predictor=True` is passed so only the encoder runs (predictor not needed)

**Reads:** `annotations/segments_verbs_timed_clean.csv` + pre-saved V-JEPA2 embeddings

**Comparison baseline:** NB4 (DinoTXT + DINOv2/v3) and NB5 Stage 1 (V-JEPA2 video-only)

## 0. Install & imports

In [1]:
!pip install -q transformers accelerate decord

In [2]:
import os, json
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
from torch.utils.data import Dataset, DataLoader
from decord import VideoReader, cpu
from transformers import AutoModel, AutoVideoProcessor

with open('/home/ubuntu/youcook2-videos/paths.json') as f:
    PATHS = json.load(f)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device : {DEVICE}")
if DEVICE.type == 'cuda':
    print(f"GPU    : {torch.cuda.get_device_name(0)}")

Device : cuda
GPU    : NVIDIA A10


## 1. Load data

In [3]:
df = pd.read_csv('/home/ubuntu/youcook2-videos/annotations/segments_verbs_timed_clean.csv')

verb_vocab = sorted(df['extracted_verb'].unique())
label2id   = {v: i for i, v in enumerate(verb_vocab)}
id2label   = {i: v for v, i in label2id.items()}
n_verbs    = len(verb_vocab)
df['verb_label'] = df['extracted_verb'].map(label2id)

df_train = df[df['subset'] == 'training'].reset_index(drop=True)
df_val   = df[df['subset'] == 'validation'].reset_index(drop=True)

print(f"Total segments : {len(df):,}")
print(f"Verb classes   : {n_verbs}")
print(f"Train          : {len(df_train):,}")
print(f"Val            : {len(df_val):,}")
print(f"Columns        : {df.columns.tolist()}")

Total segments : 13,003
Verb classes   : 282
Train          : 9,735
Val            : 3,268
Columns        : ['caption', 'video_id', 'segment_id', 'extracted_verb', 'masked_caption', 'start_sec', 'end_sec', 'subset', 'verb_label']


## 2. Load V-JEPA2 encoder (backbone only, no classification head)

We use `AutoModel` (not `VJEPA2ForVideoClassification`) because we want raw encoder
outputs, not logits. `skip_predictor=True` skips the mask-denoising predictor —
we only need the ViT encoder's `last_hidden_state`.

In [5]:
from transformers import VJEPA2ForVideoClassification, VJEPA2VideoProcessor

MODEL_NAME = 'facebook/vjepa2-vitl-fpc16-256-ssv2'

processor = VJEPA2VideoProcessor.from_pretrained(MODEL_NAME)

# Load full model then extract just the encoder — same as NB5's model.vjepa2
_full_model    = VJEPA2ForVideoClassification.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float32,
    ignore_mismatched_sizes=True,
).to(DEVICE)

vjepa2_encoder = _full_model.vjepa2  # the bare ViT encoder, no classification head
del _full_model                       # free the head weights from memory

for param in vjepa2_encoder.parameters():
    param.requires_grad = False
vjepa2_encoder.eval()

FRAMES_PER_CLIP = vjepa2_encoder.config.frames_per_clip
print(f"Frames per clip : {FRAMES_PER_CLIP}")
print(f"Hidden size     : {vjepa2_encoder.config.hidden_size}")

video_preprocessor_config.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.50G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/652 [00:00<?, ?it/s]

Frames per clip : 16
Hidden size     : 1024


## 3. DEBUG — Inspect V-JEPA2 output shape on a real segment

Uses the first real segment from train_ds (same dataset class as NB5) so the
embedding statistics are meaningful rather than random noise.
Also checks whether V-JEPA2 has a CLS token — this determines the pooling strategy.
Run this cell and read the output carefully before proceeding.

In [ ]:
from decord import VideoReader, cpu
from torch.utils.data import Dataset

FRAMES_PER_CLIP = vjepa2_encoder.config.frames_per_clip  # read from model
CROP_SIZE       = vjepa2_encoder.config.crop_size

print(f"Model expects: {FRAMES_PER_CLIP} frames per clip, {CROP_SIZE}px crop")

# ── Load one real segment from disk (same logic as NB5 YouCookVerbDataset) ────
row       = df_train.iloc[0]
vid_path  = os.path.join(PATHS['videos'], f"{row['video_id']}.mp4")
vr        = VideoReader(vid_path, ctx=cpu(0))
fps       = vr.get_avg_fps()
start_idx = max(0, int(row['start_sec'] * fps))
end_idx   = min(len(vr) - 1, int(row['end_sec'] * fps))
indices   = np.linspace(start_idx, end_idx, FRAMES_PER_CLIP, dtype=int)
frames    = vr.get_batch(indices).asnumpy()           # (T, H, W, 3)
clip      = torch.from_numpy(frames).permute(0, 3, 1, 2)  # (T, C, H, W)

print(f"\nSegment      : {row['segment_id']}")
print(f"Verb         : {row['extracted_verb']}")
print(f"Caption      : {row['masked_caption']}")
print(f"Raw clip     : {clip.shape}  dtype={clip.dtype}")

# ── Run through processor and encoder ─────────────────────────────────────────
real_inputs = processor(clip, return_tensors='pt').to(DEVICE)

print("\n=== Processor output ===")
for k, v in real_inputs.items():
    print(f"  {k}: {v.shape}  dtype={v.dtype}")

with torch.no_grad():
    real_out = vjepa2_encoder(**real_inputs, skip_predictor=True)

B, N, D = real_out.last_hidden_state.shape
print(f"\n=== Encoder output ===")
print(f"  last_hidden_state : {real_out.last_hidden_state.shape}")
print(f"  B={B}  N={N} tokens  D={D} hidden dim")
print(f"  Expected N = (T/tubelet) * (H/patch) * (W/patch)")
print(f"            = ({FRAMES_PER_CLIP}/2) * ({CROP_SIZE}/16) * ({CROP_SIZE}/16)")
print(f"            = {(FRAMES_PER_CLIP//2) * (CROP_SIZE//16) * (CROP_SIZE//16)}")

# ── CLS token check ───────────────────────────────────────────────────────────
print("\n=== CLS token check ===")
has_cls = hasattr(vjepa2_encoder, 'cls_token') or \
          any('cls_token' in name for name, _ in vjepa2_encoder.named_parameters())
print(f"  Has cls_token parameter : {has_cls}")

# Check if token[0] looks structurally different from other tokens
# (a real CLS token tends to have very different norm from patch tokens)
tok0_norm   = real_out.last_hidden_state[0, 0, :].norm().item()
other_norms = real_out.last_hidden_state[0, 1:, :].norm(dim=-1)
print(f"  token[0] L2 norm         : {tok0_norm:.4f}")
print(f"  token[1:] mean L2 norm   : {other_norms.mean().item():.4f}")
print(f"  token[1:] std  L2 norm   : {other_norms.std().item():.4f}")
print()
if not has_cls:
    print("  ➜ No CLS token found. All tokens are spatiotemporal patch tokens.")
else:
    print("  ➜ CLS token present — token[0] is a dedicated global summary token.")
    print("    You could use token[0] alone OR mean-pool all tokens.")

Model expects: 16 frames per clip, 256px crop

Segment      : --bv0V6ZjWI_000
Verb         : crush
Caption      : [MASK] and chop the garlic
Raw clip     : torch.Size([16, 3, 720, 1280])  dtype=torch.uint8

=== Processor output ===
  pixel_values_videos: torch.Size([1, 16, 3, 256, 256])  dtype=torch.float32

=== Encoder output ===
  last_hidden_state : torch.Size([1, 2048, 1024])
  B=1  N=2048 tokens  D=1024 hidden dim
  Expected N = (T/tubelet) * (H/patch) * (W/patch)
            = (16/2) * (256/16) * (256/16)
            = 2048

=== CLS token check ===
  Has cls_token parameter : False
  token[0] L2 norm         : 104.6361
  token[1:] mean L2 norm   : 95.9355
  token[1:] std  L2 norm   : 5.8312

  ➜ No CLS token found. All tokens are spatiotemporal patch tokens.
    Mean pooling across all N tokens is the correct approach.


In [9]:
# Print the full model structure so we can see the real attribute names
for name, module in _full_model.named_children():
    print(f"{name}: {type(module).__name__}")

print("\n--- Full named modules (2 levels deep) ---")
for name, module in _full_model.named_modules():
    depth = name.count('.')
    if depth <= 2:
        print(f"{'  ' * depth}{name}: {type(module).__name__}")

vjepa2: VJEPA2Model
pooler: VJEPA2AttentivePooler
classifier: Linear

--- Full named modules (2 levels deep) ---
: VJEPA2ForVideoClassification
vjepa2: VJEPA2Model
  vjepa2.encoder: VJEPA2Encoder
    vjepa2.encoder.embeddings: VJEPA2Embeddings
    vjepa2.encoder.layer: ModuleList
    vjepa2.encoder.layernorm: LayerNorm
  vjepa2.predictor: VJEPA2Predictor
    vjepa2.predictor.embeddings: VJEPA2PredictorEmbeddings
    vjepa2.predictor.layer: ModuleList
    vjepa2.predictor.layernorm: LayerNorm
    vjepa2.predictor.proj: Linear
pooler: VJEPA2AttentivePooler
  pooler.cross_attention_layer: VJEPA2PoolerCrossAttentionLayer
    pooler.cross_attention_layer.layer_norm1: LayerNorm
    pooler.cross_attention_layer.cross_attn: VJEPA2PoolerCrossAttention
    pooler.cross_attention_layer.layer_norm2: LayerNorm
    pooler.cross_attention_layer.mlp: VJEPA2MLP
  pooler.self_attention_layers: ModuleList
    pooler.self_attention_layers.0: VJEPA2PoolerSelfAttentionLayer
    pooler.self_attention_layers.

In [13]:
import sys, os

DINOV2_REPO = os.path.join(PATHS['repos'], 'dinov2')
print(f"Looking for dinov2 repo at: {DINOV2_REPO}")
print(f"Exists: {os.path.exists(DINOV2_REPO)}")

# Add to path if not already there
if DINOV2_REPO not in sys.path:
    sys.path.insert(0, DINOV2_REPO)

print(f"sys.path[0]: {sys.path[0]}")

# Verify the module is findable
import importlib.util
spec = importlib.util.find_spec('dinov2')
print(f"dinov2 found at: {spec.origin if spec else 'NOT FOUND'}")

Looking for dinov2 repo at: /home/ubuntu/youcook2-videos/repos/dinov2
Exists: True
sys.path[0]: /home/ubuntu/youcook2-videos/repos/dinov2
dinov2 found at: /home/ubuntu/youcook2-videos/repos/dinov2/dinov2/__init__.py


In [14]:
from dinov2.hub.dinotxt import dinov2_vitl14_reg4_dinotxt_tet1280d20h24l, get_tokenizer

dinotxt   = dinov2_vitl14_reg4_dinotxt_tet1280d20h24l().to(DEVICE)
tokenizer = get_tokenizer()
dinotxt.eval()

for p in dinotxt.parameters():
    p.requires_grad = False

print("DinoTXT loaded ✓")

/home/ubuntu/youcook2-videos/repos/dinov2/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/home/ubuntu/youcook2-videos/repos/dinov2/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/home/ubuntu/youcook2-videos/repos/dinov2/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


Downloading: "https://dl.fbaipublicfiles.com/dinov2/dinov2_vitl14/dinov2_vitl14_reg4_pretrain.pth" to /home/ubuntu/.cache/torch/hub/checkpoints/dinov2_vitl14_reg4_pretrain.pth


100%|██████████| 1.13G/1.13G [00:03<00:00, 348MB/s]


Downloading: "https://dl.fbaipublicfiles.com/dinov2/dinov2_vitl14/dinov2_vitl14_reg4_dinotxt_tet1280d20h24l_vision_head.pth" to /home/ubuntu/.cache/torch/hub/checkpoints/dinov2_vitl14_reg4_dinotxt_tet1280d20h24l_vision_head.pth


100%|██████████| 96.6M/96.6M [00:00<00:00, 331MB/s]


Downloading: "https://dl.fbaipublicfiles.com/dinov2/dinov2_vitl14/dinov2_vitl14_reg4_dinotxt_tet1280d20h24l_text_encoder.pth" to /home/ubuntu/.cache/torch/hub/checkpoints/dinov2_vitl14_reg4_dinotxt_tet1280d20h24l_text_encoder.pth


100%|██████████| 2.00G/2.00G [01:03<00:00, 34.0MB/s]


DinoTXT loaded ✓


In [16]:
print("=== DinoTXT top-level children ===")
for name, module in dinotxt.named_children():
    print(f"  {name}: {type(module).__name__}")

print("\n=== DinoTXT named parameters (first 10) ===")
for i, (name, param) in enumerate(dinotxt.named_parameters()):
    print(f"  {name}: {param.shape}")
    if i >= 9:
        print("  ...")
        break

=== DinoTXT top-level children ===
  visual_model: VisionTower
  text_model: TextTower

=== DinoTXT named parameters (first 10) ===
  logit_scale: torch.Size([1])
  visual_model.backbone.model.cls_token: torch.Size([1, 1, 1024])
  visual_model.backbone.model.pos_embed: torch.Size([1, 1370, 1024])
  visual_model.backbone.model.register_tokens: torch.Size([1, 4, 1024])
  visual_model.backbone.model.mask_token: torch.Size([1, 1024])
  visual_model.backbone.model.patch_embed.proj.weight: torch.Size([1024, 3, 14, 14])
  visual_model.backbone.model.patch_embed.proj.bias: torch.Size([1024])
  visual_model.backbone.model.blocks.0.norm1.weight: torch.Size([1024])
  visual_model.backbone.model.blocks.0.norm1.bias: torch.Size([1024])
  visual_model.backbone.model.blocks.0.attn.qkv.weight: torch.Size([3072, 1024])
  ...


In [20]:
encoder_dim    = _full_model.vjepa2.encoder.layernorm.weight.shape[0]
pooler_out_dim = _full_model.classifier.in_features  # what classifier expects = pooler output dim

In [21]:
print("=== DinoTXT vision tower ===")
for name, module in dinotxt.visual_model.named_children():
    print(f"  {name}: {type(module).__name__}")

print("\n=== DinoTXT text tower ===")
for name, module in dinotxt.text_model.named_children():
    print(f"  {name}: {type(module).__name__}")

# Read dims directly from weights
cls_token_dim    = dinotxt.visual_model.backbone.model.cls_token.shape[-1]
patch_embed_dim  = dinotxt.visual_model.backbone.model.patch_embed.proj.weight.shape[0]

print(f"\n=== Key dimensions ===")
print(f"  DinoTXT CLS token dim        : {cls_token_dim}")
print(f"  DinoTXT patch embed dim      : {patch_embed_dim}")
print(f"  DinoTXT concat input (CLS+mean): {cls_token_dim + patch_embed_dim}")

print(f"\n=== V-JEPA2 vs DinoTXT ===")
print(f"  V-JEPA2 encoder hidden dim   : {encoder_dim}")
print(f"  V-JEPA2 pooler output dim    : {pooler_out_dim}")
print(f"  DinoTXT vision backbone dim  : {cls_token_dim}")
print(f"  Encoder matches DinoTXT?     : {encoder_dim == cls_token_dim}")
print(f"  Pooler matches DinoTXT?      : {pooler_out_dim == cls_token_dim}")

=== DinoTXT vision tower ===
  backbone: DINOv2Wrapper
  head: VisionHead

=== DinoTXT text tower ===
  backbone: TextTransformer
  head: TextHead

=== Key dimensions ===
  DinoTXT CLS token dim        : 1024
  DinoTXT patch embed dim      : 1024
  DinoTXT concat input (CLS+mean): 2048

=== V-JEPA2 vs DinoTXT ===
  V-JEPA2 encoder hidden dim   : 1024
  V-JEPA2 pooler output dim    : 1024
  DinoTXT vision backbone dim  : 1024
  Encoder matches DinoTXT?     : True
  Pooler matches DinoTXT?      : True


In [29]:
# No need to run anything — we can read the answer directly from the source
print("=== DinoTXT VisionTower forward analysis ===")
print(f"  use_class_token      : {dinotxt.visual_model.use_class_token}")
print(f"  use_patch_tokens     : {dinotxt.visual_model.use_patch_tokens}")
print(f"  patch_tokens_pooler  : {dinotxt.visual_model.patch_tokens_pooler_type}")
print(f"  backbone embed_dim   : {dinotxt.visual_model.backbone.embed_dim}")

cls_dim   = dinotxt.visual_model.backbone.embed_dim if dinotxt.visual_model.use_class_token else 0
patch_dim = dinotxt.visual_model.backbone.embed_dim if dinotxt.visual_model.use_patch_tokens else 0
final_dim = cls_dim + patch_dim

print(f"\n  CLS contribution     : {cls_dim}")
print(f"  Patch contribution   : {patch_dim}")
print(f"  Final concat dim     : {final_dim}  ← this is what DinoTXT vision side outputs")

print(f"\n=== V-JEPA2 vs DinoTXT final answer ===")
encoder_dim    = _full_model.vjepa2.encoder.layernorm.weight.shape[0]
pooler_out_dim = _full_model.classifier.in_features
print(f"  V-JEPA2 encoder mean pool dim : {encoder_dim}")
print(f"  V-JEPA2 attentive pooler dim  : {pooler_out_dim}")
print(f"  DinoTXT vision output dim     : {final_dim}")
print(f"  Mean pool matches DinoTXT?    : {encoder_dim == final_dim}")
print(f"  Attentive pooler matches?     : {pooler_out_dim == final_dim}")

=== DinoTXT VisionTower forward analysis ===
  use_class_token      : True
  use_patch_tokens     : True
  patch_tokens_pooler  : mean
  backbone embed_dim   : 1024

  CLS contribution     : 1024
  Patch contribution   : 1024
  Final concat dim     : 2048  ← this is what DinoTXT vision side outputs

=== V-JEPA2 vs DinoTXT final answer ===
  V-JEPA2 encoder mean pool dim : 1024
  V-JEPA2 attentive pooler dim  : 1024
  DinoTXT vision output dim     : 2048
  Mean pool matches DinoTXT?    : False
  Attentive pooler matches?     : False


In [31]:
# Check TextHead's full structure before we touch anything
print("=== TextHead architecture ===")
for name, module in dinotxt.text_model.head.named_children():
    print(f"  {name}: {type(module).__name__}")

print("\n=== TextHead parameters ===")
for name, param in dinotxt.text_model.head.named_parameters():
    print(f"  {name}: {param.shape}")

print("\n=== TextModel children ===")
for name, module in dinotxt.text_model.named_children():
    print(f"  {name}: {type(module).__name__}")

# Check what methods tokenizer has
print("\n=== Tokenizer type and methods ===")
print(f"  type: {type(tokenizer)}")
print(f"  methods: {[m for m in dir(tokenizer) if not m.startswith('_')]}")

=== TextHead architecture ===
  ln_final: Identity
  block_list: ModuleList
  linear_projection: Linear

=== TextHead parameters ===
  linear_projection.weight: torch.Size([2048, 1280])

=== TextModel children ===
  backbone: TextTransformer
  head: TextHead

=== Tokenizer type and methods ===
  type: <class 'dinov2.hub.text.tokenizer.Tokenizer'>
  methods: ['bpe', 'bpe_ranks', 'byte_decoder', 'byte_encoder', 'cache', 'decode', 'decoder', 'encode', 'encoder', 'pat', 'tokenize']


In [32]:
print("=== Original linear_projection bias check ===")
print(f"  weight shape : {dinotxt.text_model.head.linear_projection.weight.shape}")
print(f"  has bias     : {dinotxt.text_model.head.linear_projection.bias is not None}")
if dinotxt.text_model.head.linear_projection.bias is not None:
    print(f"  bias shape   : {dinotxt.text_model.head.linear_projection.bias.shape}")

=== Original linear_projection bias check ===
  weight shape : torch.Size([2048, 1280])
  has bias     : False


In [34]:
# Freeze V-JEPA2 encoder explicitly
for param in _full_model.vjepa2.parameters():
    param.requires_grad = False

# Confirm
vjepa2_trainable = sum(p.numel() for p in _full_model.vjepa2.parameters() if p.requires_grad)
print(f"V-JEPA2 trainable params : {vjepa2_trainable}  (must be 0)")
assert vjepa2_trainable == 0, "V-JEPA2 encoder must be fully frozen"
print("✓ V-JEPA2 frozen")

V-JEPA2 trainable params : 0  (must be 0)
✓ V-JEPA2 frozen


In [35]:
import torch.nn as nn

# ── Step 1: Reinitialise only the linear_projection to output 1024-dim ────────
# Original: Linear(1280 → 2048) to match DINOv2's vision space
# New:      Linear(1280 → 1024) to match V-JEPA2's vision space
dinotxt.text_model.head.linear_projection = nn.Linear(1280, 1024, bias=False).to(DEVICE)

# ── Step 2: Freeze everything except the new projection ───────────────────────
for name, param in dinotxt.named_parameters():
    if 'text_model.head.linear_projection' in name:
        param.requires_grad = True
    else:
        param.requires_grad = False

trainable = [(n, p.shape) for n, p in dinotxt.named_parameters() if p.requires_grad]
frozen    = sum(p.numel() for p in dinotxt.parameters() if not p.requires_grad)
print("=== Trainable parameters ===")
for name, shape in trainable:
    print(f"  {name}: {shape}")
print(f"\nFrozen params  : {frozen/1e6:.1f}M")
print(f"Trainable total: {sum(p.numel() for p in dinotxt.parameters() if p.requires_grad):,}")

# ── Step 3: Confirm V-JEPA2 encoder is also fully frozen ──────────────────────
vjepa2_trainable = sum(p.numel() for p in _full_model.vjepa2.parameters() if p.requires_grad)
print(f"\nV-JEPA2 trainable params : {vjepa2_trainable}  (must be 0)")
assert vjepa2_trainable == 0, "V-JEPA2 encoder must be fully frozen"
print("✓ V-JEPA2 frozen")

# ── Step 4: Define text encoding function using real tokenizer API ─────────────
def encode_text(captions):
    """
    Encode a list of captions through DinoTXT's text tower.
    Returns (B, 1024) normalised text embeddings.
    """
    # tokenizer.encode returns a list of token id lists
    tokens = [tokenizer.encode(cap) for cap in captions]
    max_len = max(len(t) for t in tokens)
    # pad to same length
    padded  = [t + [0] * (max_len - len(t)) for t in tokens]
    token_tensor = torch.tensor(padded, dtype=torch.long).to(DEVICE)

    text_emb = dinotxt.text_model(token_tensor)   # (B, 1024) after new projection
    text_emb = text_emb / text_emb.norm(dim=-1, keepdim=True)  # L2 normalise
    return text_emb


# ── Step 5: DEBUG — confirm output dim matches V-JEPA2 ────────────────────────
with torch.no_grad():
    test_emb = encode_text(["crush the garlic", "slice the onion"])
    print(f"\n=== Text embedding debug ===")
    print(f"  shape  : {test_emb.shape}   (expected: [2, 1024])")
    print(f"  mean   : {test_emb.mean().item():.4f}")
    print(f"  std    : {test_emb.std().item():.4f}")
    print(f"  norm[0]: {test_emb[0].norm().item():.4f}  (should be ~1.0 after L2 norm)")
    assert test_emb.shape[-1] == 1024, f"Expected 1024-dim, got {test_emb.shape[-1]}"
    print("  ✓ Dimension matches V-JEPA2 encoder output")

=== Trainable parameters ===
  text_model.head.linear_projection.weight: torch.Size([1024, 1280])

Frozen params  : 865.2M
Trainable total: 1,310,720

V-JEPA2 trainable params : 0  (must be 0)
✓ V-JEPA2 frozen

=== Text embedding debug ===
  shape  : torch.Size([2, 1024])   (expected: [2, 1024])
  mean   : 0.0014
  std    : 0.0312
  norm[0]: 1.0000  (should be ~1.0 after L2 norm)
  ✓ Dimension matches V-JEPA2 encoder output


In [36]:
from decord import VideoReader, cpu
from tqdm.auto import tqdm

EMB_DIR = os.path.join(PATHS['embeddings'], 'vjepa2_dinotxt')
os.makedirs(EMB_DIR, exist_ok=True)
print(f"Saving embeddings to: {EMB_DIR}")

# Use bare encoder from the full model — no attentive pooler, no classifier
encoder = _full_model.vjepa2
encoder.eval()

FRAMES_PER_CLIP = _full_model.config.frames_per_clip

skipped  = 0
existing = 0

with torch.no_grad():
    for _, row in tqdm(df.iterrows(), total=len(df), desc="Extracting embeddings"):
        out_path = os.path.join(EMB_DIR, f"{row['segment_id']}.pt")
        if os.path.exists(out_path):
            existing += 1
            continue

        vid_path = os.path.join(PATHS['videos'], f"{row['video_id']}.mp4")
        try:
            vr        = VideoReader(vid_path, ctx=cpu(0))
            fps       = vr.get_avg_fps()
            start_idx = max(0, int(row['start_sec'] * fps))
            end_idx   = min(len(vr) - 1, int(row['end_sec'] * fps))
            indices   = np.linspace(start_idx, end_idx, FRAMES_PER_CLIP, dtype=int)
            frames    = vr.get_batch(indices).asnumpy()
            clip      = torch.from_numpy(frames).permute(0, 3, 1, 2)  # (T, C, H, W)

            inputs  = processor(clip, return_tensors='pt')
            inputs  = {k: v.to(DEVICE) for k, v in inputs.items()}

            out     = encoder(**inputs, skip_predictor=True)
            emb     = out.last_hidden_state.mean(dim=1).squeeze(0).cpu()  # (1024,)

            torch.save(emb, out_path)

        except Exception as e:
            print(f"  ERROR {row['segment_id']}: {e}")
            skipped += 1

saved = len([f for f in os.listdir(EMB_DIR) if f.endswith('.pt')])
print(f"\nDone.")
print(f"  Saved    : {saved:,}")
print(f"  Skipped  : {skipped:,}  (load errors)")
print(f"  Existing : {existing:,}  (already cached)")

# Spot check
sample = torch.load(os.path.join(EMB_DIR, os.listdir(EMB_DIR)[0]))
print(f"\nSpot check — shape: {sample.shape}  mean: {sample.mean():.4f}  std: {sample.std():.4f}")
assert sample.shape == (1024,), f"Expected (1024,), got {sample.shape}"
print("✓ Embedding shape correct")

Saving embeddings to: /home/ubuntu/youcook2-videos/embeddings/dinov2/vjepa2_dinotxt


Extracting embeddings:   0%|          | 0/13003 [00:00<?, ?it/s]

[h264 @ 0x49c21a00] mmco: unref short failure
[h264 @ 0x49c21a00] mmco: unref short failure
[h264 @ 0x4ae1f380] mmco: unref short failure
[h264 @ 0x4ae1f380] mmco: unref short failure
[h264 @ 0x48e73f80] mmco: unref short failure
[h264 @ 0x48e73f80] mmco: unref short failure
[h264 @ 0x49f28f80] mmco: unref short failure
[h264 @ 0x128482b00] mmco: unref short failure
[h264 @ 0x128482b00] mmco: unref short failure
[h264 @ 0x6aaa1a80] mmco: unref short failure
[h264 @ 0x483be500] mmco: unref short failure
[h264 @ 0x483be500] mmco: unref short failure
[h264 @ 0x1317ce5c0] mmco: unref short failure
[h264 @ 0x1317ce5c0] mmco: unref short failure
[h264 @ 0x981ee300] mmco: unref short failure
[h264 @ 0x981ee300] mmco: unref short failure
[h264 @ 0x981ee300] mmco: unref short failure
[h264 @ 0x981ee300] mmco: unref short failure
[h264 @ 0x146e81d40] mmco: unref short failure
[h264 @ 0x6bc31580] mmco: unref short failure
[h264 @ 0x11ee53500] mmco: unref short failure
[h264 @ 0x144f2bec0] mmco: u


Done.
  Saved    : 13,003
  Skipped  : 0  (load errors)
  Existing : 0  (already cached)

Spot check — shape: torch.Size([1024])  mean: 0.0204  std: 1.6075
✓ Embedding shape correct


In [37]:
print(inspect.getsource(dinotxt.text_model.__class__.forward))

    def forward(self, token_indices: Tensor) -> Tensor:
        text_tokens = self.backbone(token_indices)
        text_tokens = self.head(text_tokens)
        if self.tokens_pooler_type == "first":
            features = text_tokens[:, 0]
        elif self.tokens_pooler_type == "last":
            features = text_tokens[:, -1]
        elif self.tokens_pooler_type == "argmax":
            assert token_indices is not None
            features = text_tokens[torch.arange(text_tokens.shape[0]), token_indices.argmax(dim=-1)]
        else:
            raise ValueError(f"Unknown text tokens pooler type: {self.pooler_type}")
        return features



In [38]:
# Check if dinotxt has logit_scale and what forward returns
print("=== logit_scale ===")
print(f"  has logit_scale : {hasattr(dinotxt, 'logit_scale')}")
if hasattr(dinotxt, 'logit_scale'):
    print(f"  logit_scale     : {dinotxt.logit_scale}")

print("\n=== dinotxt.forward signature ===")
print(inspect.getsource(dinotxt.__class__.forward))

print("\n=== tokenizer.tokenize ===")
print(inspect.getsource(tokenizer.tokenize))

=== logit_scale ===
  has logit_scale : True
  logit_scale     : Parameter containing:
tensor([2.6593], device='cuda:0')

=== dinotxt.forward signature ===
    def forward(
        self,
        image: Tensor,
        text: Tensor,
    ) -> Tuple[Tensor, Tensor, Tensor]:

        text_features = self.encode_text(text, normalize=True)
        image_features = self.encode_image(image, normalize=True)
        return image_features, text_features, self.logit_scale.exp()


=== tokenizer.tokenize ===
    def tokenize(self, texts: Union[str, List[str]], context_length: int = 77) -> torch.LongTensor:
        """
        Returns the tokenized representation of given input string(s)

        Parameters
        ----------
        texts : Union[str, List[str]]
            An input string or a list of input strings to tokenize
        context_length : int
            The context length to use; all CLIP models use 77 as the context length

        Returns
        -------
        A two-dimensional te

In [39]:
print(inspect.getsource(dinotxt.encode_text))
print(inspect.getsource(dinotxt.encode_image))

    def encode_text(self, text: Tensor, normalize: bool = False) -> Tensor:
        """
        Encode a text input into a vector descriptor.

        Args:
            text (Tensor): Tensor of shape `(batch_size, seq_len)` containing token indices.
            normalize (bool, optional): Whether to normalize the output vectors. Default is False.
                Text features should be normalized before comparing them with image features:
        Returns:
            Tensor: Tensor of shape `(batch_size, embed_dim)` containing the text features.
                As a consequence of the training procedure, assume that the first half of the tensor corresponds
                to global image features and the second half to pooled patch features.
        """
        features = self.text_model(text)
        return F.normalize(features, dim=-1) if normalize else features

    def encode_image(
        self,
        image: Tensor,
        normalize: bool = False,
    ) -> Tensor:
        """
 

In [40]:
# ── Explicitly freeze everything except linear_projection ─────────────────────
for name, param in dinotxt.named_parameters():
    param.requires_grad = 'text_model.head.linear_projection' in name

# Also freeze V-JEPA2 encoder
for param in _full_model.vjepa2.parameters():
    param.requires_grad = False

# Confirm
trainable = [(n, p.shape) for n, p in dinotxt.named_parameters() if p.requires_grad]
print("=== Trainable parameters ===")
for name, shape in trainable:
    print(f"  {name}: {shape}")
assert len(trainable) == 1, f"Expected 1 trainable param group, got {len(trainable)}"
print(f"✓ Only linear_projection is trainable")
print(f"✓ V-JEPA2 frozen: {sum(p.numel() for p in _full_model.vjepa2.parameters() if p.requires_grad) == 0}")

=== Trainable parameters ===
  text_model.head.linear_projection.weight: torch.Size([1024, 1280])
✓ Only linear_projection is trainable
✓ V-JEPA2 frozen: True


In [41]:
import os
config_files = []
for root, dirs, files in os.walk(PATHS['repos']):
    for f in files:
        if f.endswith('.yaml') and 'dinotxt' in f.lower():
            config_files.append(os.path.join(root, f))
print('\n'.join(config_files))

In [43]:
import math
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

# ── Explicitly freeze everything except linear_projection ─────────────────────
for name, param in dinotxt.named_parameters():
    param.requires_grad = 'text_model.head.linear_projection' in name

for param in _full_model.vjepa2.parameters():
    param.requires_grad = False

trainable = [(n, p.shape) for n, p in dinotxt.named_parameters() if p.requires_grad]
print("=== Trainable parameters ===")
for name, shape in trainable:
    print(f"  {name}: {shape}")
assert len(trainable) == 1, f"Expected 1 trainable param group, got {len(trainable)}"
vjepa2_trainable = sum(p.numel() for p in _full_model.vjepa2.parameters() if p.requires_grad)
assert vjepa2_trainable == 0, "V-JEPA2 must be fully frozen"
print(f"✓ Only linear_projection is trainable")
print(f"✓ V-JEPA2 fully frozen")

# ── Dataset ───────────────────────────────────────────────────────────────────
class VJEPA2DinoTXTDataset(Dataset):
    def __init__(self, df, emb_dir):
        self.records = []
        missing = 0
        for _, row in df.iterrows():
            emb_path = os.path.join(emb_dir, f"{row['segment_id']}.pt")
            if not os.path.exists(emb_path):
                missing += 1
                continue
            self.records.append({
                'emb_path'      : emb_path,
                'masked_caption': row['masked_caption'],
                'verb_label'    : int(row['verb_label']),
            })
        if missing:
            print(f"  WARNING: {missing} segments missing embeddings")

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        r         = self.records[idx]
        video_emb = torch.load(r['emb_path'], map_location='cpu')  # (1024,)
        return video_emb, r['masked_caption'], r['verb_label']


def collate_fn(batch):
    video_embs, captions, labels = zip(*batch)
    return (
        torch.stack(video_embs),
        list(captions),
        torch.tensor(labels, dtype=torch.long),
    )


train_ds = VJEPA2DinoTXTDataset(df_train, EMB_DIR)
val_ds   = VJEPA2DinoTXTDataset(df_val,   EMB_DIR)

# NUM_WORKERS=0 avoids Python 3.12 multiprocessing cleanup errors
BATCH_SIZE  = 32
NUM_WORKERS = 0

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          collate_fn=collate_fn, num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          collate_fn=collate_fn, num_workers=NUM_WORKERS, pin_memory=True)

print(f"\nTrain samples : {len(train_ds):,}")
print(f"Val samples   : {len(val_ds):,}")
print(f"Train batches : {len(train_loader):,}")
print(f"Val batches   : {len(val_loader):,}")

# ── Optimizer ─────────────────────────────────────────────────────────────────
def exclude(n, p):
    return p.ndim < 2 or "bn" in n or "ln" in n or "bias" in n or "logit_scale" in n

def include(n, p):
    return not exclude(n, p)

named_parameters    = list(dinotxt.named_parameters())
gain_or_bias_params = [p for n, p in named_parameters if exclude(n, p) and p.requires_grad]
rest_params         = [p for n, p in named_parameters if include(n, p) and p.requires_grad]

NUM_EPOCHS           = 5
LR                   = 0.0007
WEIGHT_DECAY         = 0.0001
GRADIENT_ACCUM_STEPS = 4

optimizer = torch.optim.AdamW(
    [
        {'params': gain_or_bias_params, 'weight_decay': 0.0},
        {'params': rest_params,         'weight_decay': WEIGHT_DECAY},
    ],
    lr=LR,
    betas=(0.9, 0.99),
    eps=1e-8,
)

# Cosine decay from epoch 1 — no warmup
def lr_lambda(epoch):
    progress = epoch / max(1, NUM_EPOCHS)
    return 0.5 * (1 + math.cos(math.pi * progress))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

CKPT_DIR = PATHS.get('checkpoints', '/home/ubuntu/youcook2-videos/checkpoints')
os.makedirs(CKPT_DIR, exist_ok=True)

print(f"\nStarting training — {NUM_EPOCHS} epochs")
print(f"LR={LR}, weight_decay={WEIGHT_DECAY}, batch={BATCH_SIZE}x{GRADIENT_ACCUM_STEPS} (accum)")
print(f"Trainable params : {sum(p.numel() for p in dinotxt.parameters() if p.requires_grad):,}\n")

# ── Verb text embeddings for evaluation ───────────────────────────────────────
def get_verb_text_embs():
    dinotxt.eval()
    with torch.no_grad():
        verb_tokens = tokenizer.tokenize(verb_vocab).to(DEVICE)
        verb_embs   = dinotxt.encode_text(verb_tokens, normalize=True)
    return verb_embs

# ── Evaluation ────────────────────────────────────────────────────────────────
def evaluate(loader):
    dinotxt.eval()
    verb_embs = get_verb_text_embs()
    correct_top1, correct_top5, total = 0, 0, 0

    with torch.no_grad():
        for video_embs, captions, labels in loader:
            video_embs  = F.normalize(video_embs.to(DEVICE), dim=-1)
            labels      = labels.to(DEVICE)
            logit_scale = dinotxt.logit_scale.exp()
            sim         = logit_scale * video_embs @ verb_embs.T

            correct_top1 += (sim.argmax(dim=-1) == labels).sum().item()
            top5          = sim.topk(5, dim=-1).indices
            correct_top5 += (top5 == labels.unsqueeze(-1)).any(dim=-1).sum().item()
            total        += labels.size(0)

    return correct_top1 / total, correct_top5 / total

# ── Training loop ─────────────────────────────────────────────────────────────
for epoch in range(1, NUM_EPOCHS + 1):
    dinotxt.train()
    optimizer.zero_grad()
    running_loss = 0.0

    current_lr = optimizer.param_groups[0]['lr']
    print(f"Epoch {epoch}/{NUM_EPOCHS} — LR: {current_lr:.6f}")

    for step, (video_embs, captions, labels) in enumerate(
        tqdm(train_loader, desc=f"  Training"), start=1
    ):
        video_embs  = F.normalize(video_embs.to(DEVICE), dim=-1)
        text_tokens = tokenizer.tokenize(captions).to(DEVICE)
        text_embs   = dinotxt.encode_text(text_tokens, normalize=True)

        logit_scale = dinotxt.logit_scale.exp()
        sim         = logit_scale * text_embs @ video_embs.T
        targets     = torch.arange(sim.shape[0]).to(DEVICE)
        loss        = (F.cross_entropy(sim, targets) +
                       F.cross_entropy(sim.T, targets)) / 2

        loss = loss / GRADIENT_ACCUM_STEPS
        loss.backward()
        running_loss += loss.item() * GRADIENT_ACCUM_STEPS

        if step % GRADIENT_ACCUM_STEPS == 0:
            optimizer.step()
            optimizer.zero_grad()
            with torch.no_grad():
                dinotxt.logit_scale.clamp_(0, math.log(100))

    # flush leftover gradients
    if step % GRADIENT_ACCUM_STEPS != 0:
        optimizer.step()
        optimizer.zero_grad()
        with torch.no_grad():
            dinotxt.logit_scale.clamp_(0, math.log(100))

    scheduler.step()

    top1, top5 = evaluate(val_loader)
    avg_loss   = running_loss / len(train_loader)
    print(f"  Loss: {avg_loss:.4f} | "
          f"Val Top-1: {top1:.4f} | Val Top-5: {top5:.4f} | "
          f"logit_scale: {dinotxt.logit_scale.exp().item():.4f}")

    ckpt_path = os.path.join(CKPT_DIR, f'vjepa2_dinotxt_proj_epoch{epoch}.pt')
    torch.save(dinotxt.text_model.head.linear_projection.state_dict(), ckpt_path)
    print(f"  Checkpoint saved → {ckpt_path}")

print("\nTraining complete.")

=== Trainable parameters ===
  text_model.head.linear_projection.weight: torch.Size([1024, 1280])
✓ Only linear_projection is trainable
✓ V-JEPA2 fully frozen

Train samples : 9,735
Val samples   : 3,268
Train batches : 305
Val batches   : 103

Starting training — 5 epochs
LR=0.0007, weight_decay=0.0001, batch=32x4 (accum)
Trainable params : 1,310,720

Epoch 1/5 — LR: 0.000700


  Training:   0%|          | 0/305 [00:00<?, ?it/s]

  Loss: 2.6524 | Val Top-1: 0.0199 | Val Top-5: 0.0670 | logit_scale: 14.2857
  Checkpoint saved → /home/ubuntu/youcook2-videos/checkpoints/vjepa2_dinotxt_proj_epoch1.pt
Epoch 2/5 — LR: 0.000633


  Training:   0%|          | 0/305 [00:00<?, ?it/s]

  Loss: 2.2416 | Val Top-1: 0.0190 | Val Top-5: 0.0701 | logit_scale: 14.2857
  Checkpoint saved → /home/ubuntu/youcook2-videos/checkpoints/vjepa2_dinotxt_proj_epoch2.pt
Epoch 3/5 — LR: 0.000458


  Training:   0%|          | 0/305 [00:00<?, ?it/s]

  Loss: 2.0898 | Val Top-1: 0.0214 | Val Top-5: 0.0753 | logit_scale: 14.2857
  Checkpoint saved → /home/ubuntu/youcook2-videos/checkpoints/vjepa2_dinotxt_proj_epoch3.pt
Epoch 4/5 — LR: 0.000242


  Training:   0%|          | 0/305 [00:00<?, ?it/s]

  Loss: 1.9810 | Val Top-1: 0.0199 | Val Top-5: 0.0704 | logit_scale: 14.2857
  Checkpoint saved → /home/ubuntu/youcook2-videos/checkpoints/vjepa2_dinotxt_proj_epoch4.pt
Epoch 5/5 — LR: 0.000067


  Training:   0%|          | 0/305 [00:00<?, ?it/s]

  Loss: 1.9107 | Val Top-1: 0.0202 | Val Top-5: 0.0701 | logit_scale: 14.2857
  Checkpoint saved → /home/ubuntu/youcook2-videos/checkpoints/vjepa2_dinotxt_proj_epoch5.pt

Training complete.


In [45]:
# Check what columns we have available
print("DataFrame columns:", df_val.columns.tolist())
print("\nSample row:")
print(df_val.iloc[0][['segment_id', 'masked_caption', 'extracted_verb', 'verb_label']])

# Check a few masked captions to understand the pattern
print("\n=== Sample masked captions ===")
for i in range(5):
    row = df_val.iloc[i]
    print(f"  caption : {row['masked_caption']}")
    print(f"  verb    : {row['extracted_verb']}")
    print(f"  filled  : {row['masked_caption'].replace('[MASK]', row['extracted_verb'])}")
    print()

# Check best epoch checkpoint
import os
ckpts = sorted([f for f in os.listdir(CKPT_DIR) if 'vjepa2_dinotxt' in f])
print("=== Available checkpoints ===")
for c in ckpts:
    print(f"  {c}")

DataFrame columns: ['caption', 'video_id', 'segment_id', 'extracted_verb', 'masked_caption', 'start_sec', 'end_sec', 'subset', 'verb_label']

Sample row:
segment_id                                          -AwyG1JcMp8_000
masked_caption    [MASK] kimchi sausage soy sauce sesame oil gre...
extracted_verb                                              combine
verb_label                                                       46
Name: 0, dtype: object

=== Sample masked captions ===
  caption : [MASK] kimchi sausage soy sauce sesame oil green onion ginger and red pepper flakes
  verb    : combine
  filled  : combine kimchi sausage soy sauce sesame oil green onion ginger and red pepper flakes

  caption : [MASK] the filling on a wrapper and press the edges together
  verb    : place
  filled  : place the filling on a wrapper and press the edges together

  caption : [MASK] the dumplings above a pot of boiling water
  verb    : place
  filled  : place the dumplings above a pot of boiling water


In [48]:
import torch.nn.functional as F
from tqdm.auto import tqdm

# ── Load best checkpoint ───────────────────────────────────────────────────────
BEST_EPOCH = 2
ckpt_path  = os.path.join(CKPT_DIR, f'vjepa2_dinotxt_proj_epoch{BEST_EPOCH}.pt')
dinotxt.text_model.head.linear_projection.load_state_dict(
    torch.load(ckpt_path, map_location=DEVICE)
)
print(f"Loaded checkpoint: {ckpt_path}")


def run_evaluation(df_subset, desc="Evaluating"):
    dinotxt.eval()
    results = []

    with torch.no_grad():
        for _, row in tqdm(df_subset.iterrows(), total=len(df_subset), desc=desc):
            emb_path = os.path.join(EMB_DIR, f"{row['segment_id']}.pt")
            if not os.path.exists(emb_path):
                print(f"  MISSING embedding: {row['segment_id']}")
                continue

            # video embedding
            video_emb = torch.load(emb_path, map_location=DEVICE)
            video_emb = F.normalize(video_emb.unsqueeze(0), dim=-1)  # (1, 1024)

            # fill each verb into the masked caption
            filled = [row['masked_caption'].replace('[MASK]', v) for v in verb_vocab]

            # encode in chunks to avoid OOM
            CHUNK = 64
            all_text_embs = []
            for i in range(0, len(filled), CHUNK):
                chunk_tokens = tokenizer.tokenize(filled[i:i+CHUNK]).to(DEVICE)
                chunk_embs   = dinotxt.encode_text(chunk_tokens, normalize=True)
                all_text_embs.append(chunk_embs)
            text_embs = torch.cat(all_text_embs, dim=0)  # (n_verbs, 1024)

            logit_scale  = dinotxt.logit_scale.exp()
            sim          = (logit_scale * video_emb @ text_embs.T).squeeze(0)  # (n_verbs,)
            top5_indices = sim.topk(5).indices.tolist()
            top1_verb    = verb_vocab[top5_indices[0]]
            gt_verb      = row['extracted_verb']
            gt_idx       = label2id.get(gt_verb, -1)

            results.append({
                'segment_id'    : row['segment_id'],
                'masked_caption': row['masked_caption'],
                'gt_verb'       : gt_verb,
                'top1_pred'     : top1_verb,
                'top5_preds'    : [verb_vocab[i] for i in top5_indices],
                'top1_correct'  : top1_verb == gt_verb,
                'top5_correct'  : gt_idx in top5_indices,
                'filled_gt'     : row['masked_caption'].replace('[MASK]', gt_verb),
                'filled_top1'   : row['masked_caption'].replace('[MASK]', top1_verb),
            })

    return pd.DataFrame(results)


def print_results(results_df, label=""):
    top1 = results_df['top1_correct'].mean()
    top5 = results_df['top5_correct'].mean()

    print(f"\n=== {label} ===")
    print(f"  Evaluated segments : {len(results_df):,}")
    print(f"  Val Top-1          : {top1:.4f}  ({top1*100:.2f}%)")
    print(f"  Val Top-5          : {top5:.4f}  ({top5*100:.2f}%)")
    print(f"  Random baseline    : {1/len(verb_vocab):.4f}  ({100/len(verb_vocab):.2f}%)")
    print(f"  Top-1 vs random    : {top1/(1/len(verb_vocab)):.1f}x above chance")

    print(f"\n=== Sample CORRECT predictions ===")
    for _, row in results_df[results_df['top1_correct']].head(3).iterrows():
        print(f"  Caption : {row['masked_caption']}")
        print(f"  GT/Pred : {row['gt_verb']}")
        print()

    print(f"=== Sample INCORRECT predictions ===")
    for _, row in results_df[~results_df['top1_correct']].head(3).iterrows():
        print(f"  Caption : {row['masked_caption']}")
        print(f"  GT verb : {row['gt_verb']}")
        print(f"  Pred    : {row['top1_pred']}")
        print(f"  Top-5   : {row['top5_preds']}")
        print()


# ── STAGE 1: Sanity check on 4 videos ─────────────────────────────────────────
# Pick 4 unique video_ids from val set
sample_video_ids = df_val['video_id'].unique()[:4]
df_sample        = df_val[df_val['video_id'].isin(sample_video_ids)].reset_index(drop=True)

print(f"Sanity check — {len(sample_video_ids)} videos, {len(df_sample)} segments")
print(f"Video IDs: {list(sample_video_ids)}")
print(f"Segments per video:")
print(df_sample.groupby('video_id').size())

sample_results = run_evaluation(df_sample, desc="Sanity check")
print_results(sample_results, label=f"Sanity Check — {len(sample_video_ids)} videos")

# ── Pause here — confirm sanity check looks reasonable before full eval ────────
print("\n" + "="*60)
print("Sanity check complete. Results look reasonable?")
print("If yes, proceed to the next cell for full evaluation.")
print("="*60)

Loaded checkpoint: /home/ubuntu/youcook2-videos/checkpoints/vjepa2_dinotxt_proj_epoch2.pt
Sanity check — 4 videos, 31 segments
Video IDs: ['-AwyG1JcMp8', '-ErPSunMfcs', '-GlSSp5ZOCQ', '-Ju39A-G0Dk']
Segments per video:
video_id
-AwyG1JcMp8     5
-ErPSunMfcs     7
-GlSSp5ZOCQ     9
-Ju39A-G0Dk    10
dtype: int64


Sanity check:   0%|          | 0/31 [00:00<?, ?it/s]


=== Sanity Check — 4 videos ===
  Evaluated segments : 31
  Val Top-1          : 0.0000  (0.00%)
  Val Top-5          : 0.0000  (0.00%)
  Random baseline    : 0.0035  (0.35%)
  Top-1 vs random    : 0.0x above chance

=== Sample CORRECT predictions ===
=== Sample INCORRECT predictions ===
  Caption : [MASK] kimchi sausage soy sauce sesame oil green onion ginger and red pepper flakes
  GT verb : combine
  Pred    : cauliflower
  Top-5   : ['cauliflower', 'coat', 'case', 'wear', 'mince']

  Caption : [MASK] the filling on a wrapper and press the edges together
  GT verb : place
  Pred    : carrot
  Top-5   : ['carrot', 'orange', 'pepperoni', 'cucumber', 'glue']

  Caption : [MASK] the dumplings above a pot of boiling water
  GT verb : place
  Pred    : bundle
  Top-5   : ['bundle', 'skewer', 'scallian', 'appear', 'brush']


Sanity check complete. Results look reasonable?
If yes, proceed to the next cell for full evaluation.


In [49]:
# ── Debug: inspect what's happening step by step ──────────────────────────────
row = df_val.iloc[0]
print(f"Caption  : {row['masked_caption']}")
print(f"GT verb  : {row['extracted_verb']}")

# Check video embedding
emb_path  = os.path.join(EMB_DIR, f"{row['segment_id']}.pt")
video_emb = torch.load(emb_path, map_location=DEVICE)
video_emb_norm = F.normalize(video_emb.unsqueeze(0), dim=-1)
print(f"\nVideo emb shape : {video_emb.shape}")
print(f"Video emb norm  : {video_emb_norm.norm().item():.4f}  (should be 1.0)")
print(f"Video emb mean  : {video_emb.mean().item():.4f}")
print(f"Video emb std   : {video_emb.std().item():.4f}")

# Check text embeddings for a few known verbs
test_verbs    = ['combine', 'place', 'mix', 'chop', 'add']
test_filled   = [row['masked_caption'].replace('[MASK]', v) for v in test_verbs]
test_tokens   = tokenizer.tokenize(test_filled).to(DEVICE)
test_embs     = dinotxt.encode_text(test_tokens, normalize=True)

print(f"\nText emb shape  : {test_embs.shape}")
print(f"Text emb norm[0]: {test_embs[0].norm().item():.4f}  (should be 1.0)")
print(f"Text emb mean   : {test_embs.mean().item():.4f}")
print(f"Text emb std    : {test_embs.std().item():.4f}")

# Check similarity between video and these test verbs
logit_scale = dinotxt.logit_scale.exp()
sim = (logit_scale * video_emb_norm @ test_embs.T).squeeze(0)
print(f"\nLogit scale     : {logit_scale.item():.4f}")
print(f"\nSimilarity scores for known verbs:")
for v, s in zip(test_verbs, sim.tolist()):
    print(f"  {v:<10} : {s:.4f}")

# Check if linear_projection weights look reasonable
proj = dinotxt.text_model.head.linear_projection
print(f"\nlinear_projection weight stats:")
print(f"  mean : {proj.weight.mean().item():.6f}")
print(f"  std  : {proj.weight.std().item():.6f}")
print(f"  norm : {proj.weight.norm().item():.4f}")

# Check verb_vocab — are non-verbs in there?
print(f"\nFirst 20 verb_vocab entries: {verb_vocab[:20]}")
print(f"Total verb vocab size: {len(verb_vocab)}")

Caption  : [MASK] kimchi sausage soy sauce sesame oil green onion ginger and red pepper flakes
GT verb  : combine

Video emb shape : torch.Size([1024])
Video emb norm  : 1.0000  (should be 1.0)
Video emb mean  : 0.0267
Video emb std   : 1.7620

Text emb shape  : torch.Size([5, 1024])
Text emb norm[0]: 1.0000  (should be 1.0)
Text emb mean   : -0.0002
Text emb std    : 0.0313

Logit scale     : 14.2857

Similarity scores for known verbs:
  combine    : 2.3333
  place      : 2.4454
  mix        : 2.7423
  chop       : 2.3102
  add        : 2.1440

linear_projection weight stats:
  mean : -0.000003
  std  : 0.017525
  norm : 20.0644

First 20 verb_vocab entries: ['absorb', 'add', 'addre', 'adjust', 'allow', 'appear', 'apply', 'arange', 'arrange', 'attach', 'bake', 'barbecue', 'batter', 'beat', 'become', 'begin', 'blanch', 'blend', 'boil', 'break']
Total verb vocab size: 282


In [50]:
# Check if projection weights actually changed from random init
# A random init of Linear(1280, 1024) with default PyTorch init has std ~ 1/sqrt(1280) ~ 0.028
print(f"Projection weight std : {proj.weight.std().item():.6f}")
print(f"Expected random init  : {1/1280**0.5:.6f}")

# If they're the same, the weights barely moved during training
# Check by looking at gradient norms from last training step
print(f"\nDoes projection have grad? : {proj.weight.grad is not None}")
if proj.weight.grad is not None:
    print(f"Grad norm : {proj.weight.grad.norm().item():.6f}")

# Also check — was the projection actually reinitialised before training?
# The checkpoint from epoch 2 should have different weights than random init
ckpt = torch.load(ckpt_path, map_location='cpu')
print(f"\nCheckpoint weight std  : {ckpt['weight'].std().item():.6f}")
print(f"Checkpoint weight mean : {ckpt['weight'].mean().item():.6f}")
print(f"Checkpoint weight norm : {ckpt['weight'].norm().item():.4f}")

# Compare to what's currently loaded
print(f"\nCurrent weight std     : {proj.weight.std().item():.6f}")
print(f"Current weight norm    : {proj.weight.norm().item():.4f}")
print(f"Same as checkpoint?    : {torch.allclose(ckpt['weight'].to(DEVICE), proj.weight)}")

Projection weight std : 0.017525
Expected random init  : 0.027951

Does projection have grad? : False

Checkpoint weight std  : 0.017525
Checkpoint weight mean : -0.000003
Checkpoint weight norm : 20.0641

Current weight std     : 0.017525
Current weight norm    : 20.0644
Same as checkpoint?    : True


In [51]:
# Confirm the collapse by checking weight norm evolution across checkpoints
print("=== Weight norm across epochs ===")
for ep in range(1, 6):
    ckpt = torch.load(
        os.path.join(CKPT_DIR, f'vjepa2_dinotxt_proj_epoch{ep}.pt'),
        map_location='cpu'
    )
    print(f"  Epoch {ep} : std={ckpt['weight'].std().item():.6f}  norm={ckpt['weight'].norm().item():.4f}")

# Also check what random init looks like for comparison
import torch.nn as nn
random_proj = nn.Linear(1280, 1024, bias=False)
print(f"\n  Random init : std={random_proj.weight.std().item():.6f}  norm={random_proj.weight.norm().item():.4f}")

=== Weight norm across epochs ===
  Epoch 1 : std=0.017160  norm=19.6456
  Epoch 2 : std=0.017525  norm=20.0641
  Epoch 3 : std=0.017735  norm=20.3044
  Epoch 4 : std=0.017776  norm=20.3504
  Epoch 5 : std=0.017767  norm=20.3400

  Random init : std=0.016142  norm=18.4799


In [52]:
# How similar are ALL verb text embeddings to each other?
# If they cluster together, discrimination is impossible
dinotxt.eval()
with torch.no_grad():
    # encode all verbs with the same caption context
    test_caption = df_val.iloc[0]['masked_caption']
    filled       = [test_caption.replace('[MASK]', v) for v in verb_vocab]
    
    CHUNK = 64
    all_embs = []
    for i in range(0, len(filled), CHUNK):
        tokens = tokenizer.tokenize(filled[i:i+CHUNK]).to(DEVICE)
        embs   = dinotxt.encode_text(tokens, normalize=True)
        all_embs.append(embs)
    all_embs = torch.cat(all_embs, dim=0)  # (282, 1024)

    # pairwise cosine similarity between all verb embeddings
    sim_matrix = all_embs @ all_embs.T  # (282, 282)
    
    # exclude diagonal (self-similarity = 1.0)
    mask = ~torch.eye(len(verb_vocab), dtype=torch.bool, device=DEVICE)
    off_diag = sim_matrix[mask]

print(f"=== Inter-verb text embedding similarity ===")
print(f"  Mean pairwise similarity : {off_diag.mean().item():.4f}")
print(f"  Std pairwise similarity  : {off_diag.std().item():.4f}")
print(f"  Min pairwise similarity  : {off_diag.min().item():.4f}")
print(f"  Max pairwise similarity  : {off_diag.max().item():.4f}")

# Also check video-to-text similarity range
emb_path  = os.path.join(EMB_DIR, f"{df_val.iloc[0]['segment_id']}.pt")
video_emb = F.normalize(torch.load(emb_path, map_location=DEVICE).unsqueeze(0), dim=-1)
v2t_sim   = (video_emb @ all_embs.T).squeeze(0)

print(f"\n=== Video-to-text similarity range ===")
print(f"  Mean : {v2t_sim.mean().item():.4f}")
print(f"  Std  : {v2t_sim.std().item():.4f}")
print(f"  Min  : {v2t_sim.min().item():.4f}")
print(f"  Max  : {v2t_sim.max().item():.4f}")
print(f"  GT verb '{df_val.iloc[0]['extracted_verb']}' sim : {v2t_sim[label2id[df_val.iloc[0]['extracted_verb']]].item():.4f}")
print(f"  GT verb rank : {(v2t_sim > v2t_sim[label2id[df_val.iloc[0]['extracted_verb']]]).sum().item() + 1} / {len(verb_vocab)}")

=== Inter-verb text embedding similarity ===
  Mean pairwise similarity : 0.8778
  Std pairwise similarity  : 0.0536
  Min pairwise similarity  : 0.5274
  Max pairwise similarity  : 0.9846

=== Video-to-text similarity range ===
  Mean : 0.1569
  Std  : 0.0242
  Min  : 0.0763
  Max  : 0.2159
  GT verb 'combine' sim : 0.1633
  GT verb rank : 122 / 282


In [53]:
import math
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

# ── Reinitialise projection with fresh weights ────────────────────────────────
import torch.nn as nn
dinotxt.text_model.head.linear_projection = nn.Linear(1280, 1024, bias=False).to(DEVICE)
print(f"Fresh projection — std: {dinotxt.text_model.head.linear_projection.weight.std().item():.6f}")

# ── Freeze everything except linear_projection ────────────────────────────────
for name, param in dinotxt.named_parameters():
    param.requires_grad = 'text_model.head.linear_projection' in name
for param in _full_model.vjepa2.parameters():
    param.requires_grad = False

trainable = [(n, p.shape) for n, p in dinotxt.named_parameters() if p.requires_grad]
assert len(trainable) == 1
print(f"✓ Only linear_projection trainable")

# ── Dataset ───────────────────────────────────────────────────────────────────
class VJEPA2DinoTXTDataset(Dataset):
    def __init__(self, df, emb_dir):
        self.records = []
        missing = 0
        for _, row in df.iterrows():
            emb_path = os.path.join(emb_dir, f"{row['segment_id']}.pt")
            if not os.path.exists(emb_path):
                missing += 1
                continue
            self.records.append({
                'emb_path'      : emb_path,
                'masked_caption': row['masked_caption'],
                'verb_label'    : int(row['verb_label']),
            })
        if missing:
            print(f"  WARNING: {missing} segments missing embeddings")

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        r         = self.records[idx]
        video_emb = torch.load(r['emb_path'], map_location='cpu')
        return video_emb, r['masked_caption'], r['verb_label']

def collate_fn(batch):
    video_embs, captions, labels = zip(*batch)
    return (
        torch.stack(video_embs),
        list(captions),
        torch.tensor(labels, dtype=torch.long),
    )

train_ds     = VJEPA2DinoTXTDataset(df_train, EMB_DIR)
val_ds       = VJEPA2DinoTXTDataset(df_val,   EMB_DIR)
BATCH_SIZE   = 32
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          collate_fn=collate_fn, num_workers=0, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          collate_fn=collate_fn, num_workers=0, pin_memory=True)

print(f"Train: {len(train_ds):,}  Val: {len(val_ds):,}")

# ── Pre-compute ALL verb candidate embeddings per unique caption ───────────────
# For each unique masked caption, pre-fill all 282 verbs and encode them
# This is the key change — training now sees all 282 verbs, not just batch pairs

print("\nPre-computing verb candidate embeddings for all unique captions...")
unique_captions = df_train['masked_caption'].unique().tolist()
print(f"Unique train captions: {len(unique_captions):,}")

# We don't cache these because linear_projection changes every step
# Instead we pre-compute verb tokens once (the tokenization doesn't change)
print("Pre-tokenizing all verb-filled captions...")

# Build a lookup: caption → pre-tokenized tensor of all 282 filled versions
# Too large to store all at once, so we tokenize verb templates instead
# and fill dynamically — but tokenization is the slow part so cache that

# Actually just pre-tokenize bare verbs — context adds noise anyway
# This is more principled: verb semantics should drive discrimination
verb_tokens_bare = tokenizer.tokenize(verb_vocab).to(DEVICE)  # (282, 77)
print(f"Verb tokens shape: {verb_tokens_bare.shape}")

# ── Optimizer ─────────────────────────────────────────────────────────────────
NUM_EPOCHS           = 5
LR                   = 1e-4      # conservative — avoids collapse
WEIGHT_DECAY         = 0.0       # no weight decay — prevents shrinkage
GRADIENT_ACCUM_STEPS = 4

optimizer = torch.optim.AdamW(
    [p for p in dinotxt.parameters() if p.requires_grad],
    lr=LR,
    betas=(0.9, 0.99),
    eps=1e-8,
    weight_decay=WEIGHT_DECAY,
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

CKPT_DIR = PATHS.get('checkpoints', '/home/ubuntu/youcook2-videos/checkpoints')
os.makedirs(CKPT_DIR, exist_ok=True)

# ── Evaluation using bare verb tokens ─────────────────────────────────────────
def evaluate(loader):
    dinotxt.eval()
    with torch.no_grad():
        # encode all bare verbs with current projection
        verb_embs = dinotxt.encode_text(verb_tokens_bare, normalize=True)  # (282, 1024)

    correct_top1, correct_top5, total = 0, 0, 0
    with torch.no_grad():
        for video_embs, captions, labels in loader:
            video_embs = F.normalize(video_embs.to(DEVICE), dim=-1)
            labels     = labels.to(DEVICE)
            sim        = dinotxt.logit_scale.exp() * video_embs @ verb_embs.T
            correct_top1 += (sim.argmax(dim=-1) == labels).sum().item()
            top5          = sim.topk(5, dim=-1).indices
            correct_top5 += (top5 == labels.unsqueeze(-1)).any(dim=-1).sum().item()
            total        += labels.size(0)
    return correct_top1 / total, correct_top5 / total

# ── Training loop ─────────────────────────────────────────────────────────────
print(f"\nStarting training — {NUM_EPOCHS} epochs, LR={LR}, no weight decay")
print(f"Loss: CrossEntropy over all {len(verb_vocab)} verb candidates\n")

for epoch in range(1, NUM_EPOCHS + 1):
    dinotxt.train()
    optimizer.zero_grad()
    running_loss = 0.0

    print(f"Epoch {epoch}/{NUM_EPOCHS} — LR: {optimizer.param_groups[0]['lr']:.6f}")

    for step, (video_embs, captions, labels) in enumerate(
        tqdm(train_loader, desc="  Training"), start=1
    ):
        video_embs = F.normalize(video_embs.to(DEVICE), dim=-1)  # (B, 1024)
        labels     = labels.to(DEVICE)                            # (B,)

        # encode all 282 bare verbs with current projection → (282, 1024)
        verb_embs  = dinotxt.encode_text(verb_tokens_bare, normalize=True)

        # similarity: (B, 282)
        logit_scale = dinotxt.logit_scale.exp()
        sim         = logit_scale * video_embs @ verb_embs.T

        # CrossEntropy over all 282 verbs — directly optimises verb discrimination
        loss = F.cross_entropy(sim, labels) / GRADIENT_ACCUM_STEPS
        loss.backward()
        running_loss += loss.item() * GRADIENT_ACCUM_STEPS

        if step % GRADIENT_ACCUM_STEPS == 0:
            optimizer.step()
            optimizer.zero_grad()
            with torch.no_grad():
                dinotxt.logit_scale.clamp_(0, math.log(100))

    if step % GRADIENT_ACCUM_STEPS != 0:
        optimizer.step()
        optimizer.zero_grad()
        with torch.no_grad():
            dinotxt.logit_scale.clamp_(0, math.log(100))

    scheduler.step()

    top1, top5 = evaluate(val_loader)
    avg_loss   = running_loss / len(train_loader)
    print(f"  Loss: {avg_loss:.4f} | Val Top-1: {top1:.4f} | "
          f"Val Top-5: {top5:.4f} | "
          f"logit_scale: {dinotxt.logit_scale.exp().item():.4f}")

    # track weight norm to detect collapse
    proj_norm = dinotxt.text_model.head.linear_projection.weight.norm().item()
    print(f"  Projection norm: {proj_norm:.4f}")

    ckpt_path = os.path.join(CKPT_DIR, f'vjepa2_dinotxt_v2_epoch{epoch}.pt')
    torch.save(dinotxt.text_model.head.linear_projection.state_dict(), ckpt_path)
    print(f"  Checkpoint → {ckpt_path}")

print("\nTraining complete.")

Fresh projection — std: 0.016127
✓ Only linear_projection trainable
Train: 9,735  Val: 3,268

Pre-computing verb candidate embeddings for all unique captions...
Unique train captions: 9,110
Pre-tokenizing all verb-filled captions...
Verb tokens shape: torch.Size([282, 77])

Starting training — 5 epochs, LR=0.0001, no weight decay
Loss: CrossEntropy over all 282 verb candidates

Epoch 1/5 — LR: 0.000100


  Training:   0%|          | 0/305 [00:00<?, ?it/s]

  Loss: 3.5021 | Val Top-1: 0.3681 | Val Top-5: 0.6010 | logit_scale: 14.2857
  Projection norm: 18.5289
  Checkpoint → /home/ubuntu/youcook2-videos/checkpoints/vjepa2_dinotxt_v2_epoch1.pt
Epoch 2/5 — LR: 0.000090


  Training:   0%|          | 0/305 [00:00<?, ?it/s]

  Loss: 2.9435 | Val Top-1: 0.3764 | Val Top-5: 0.6206 | logit_scale: 14.2857
  Projection norm: 18.5398
  Checkpoint → /home/ubuntu/youcook2-videos/checkpoints/vjepa2_dinotxt_v2_epoch2.pt
Epoch 3/5 — LR: 0.000065


  Training:   0%|          | 0/305 [00:00<?, ?it/s]

  Loss: 2.8343 | Val Top-1: 0.3761 | Val Top-5: 0.6365 | logit_scale: 14.2857
  Projection norm: 18.5375
  Checkpoint → /home/ubuntu/youcook2-videos/checkpoints/vjepa2_dinotxt_v2_epoch3.pt
Epoch 4/5 — LR: 0.000035


  Training:   0%|          | 0/305 [00:00<?, ?it/s]

  Loss: 2.7786 | Val Top-1: 0.3794 | Val Top-5: 0.6340 | logit_scale: 14.2857
  Projection norm: 18.5325
  Checkpoint → /home/ubuntu/youcook2-videos/checkpoints/vjepa2_dinotxt_v2_epoch4.pt
Epoch 5/5 — LR: 0.000010


  Training:   0%|          | 0/305 [00:00<?, ?it/s]

  Loss: 2.7524 | Val Top-1: 0.3807 | Val Top-5: 0.6392 | logit_scale: 14.2857
  Projection norm: 18.5302
  Checkpoint → /home/ubuntu/youcook2-videos/checkpoints/vjepa2_dinotxt_v2_epoch5.pt

Training complete.


In [54]:
# Load epoch 5 checkpoint and continue with higher LR
ckpt = torch.load(
    os.path.join(CKPT_DIR, 'vjepa2_dinotxt_v2_epoch5.pt'),
    map_location=DEVICE
)
dinotxt.text_model.head.linear_projection.load_state_dict(ckpt)
print(f"Loaded epoch 5 checkpoint")
print(f"Projection norm: {dinotxt.text_model.head.linear_projection.weight.norm().item():.4f}")

# Retrain with higher LR — projection norm needs to move
NUM_EPOCHS           = 5
LR                   = 3e-4
WEIGHT_DECAY         = 0.0
GRADIENT_ACCUM_STEPS = 4

optimizer = torch.optim.AdamW(
    [p for p in dinotxt.parameters() if p.requires_grad],
    lr=LR,
    betas=(0.9, 0.99),
    eps=1e-8,
    weight_decay=WEIGHT_DECAY,
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

print(f"Continuing training — {NUM_EPOCHS} more epochs, LR={LR}")
print(f"Starting from Val Top-1: 0.3807  Val Top-5: 0.6392\n")

Loaded epoch 5 checkpoint
Projection norm: 18.5302
Continuing training — 5 more epochs, LR=0.0003
Starting from Val Top-1: 0.3807  Val Top-5: 0.6392



In [55]:
import math
import torch.nn.functional as F
from tqdm.auto import tqdm

# ── Load best checkpoint ───────────────────────────────────────────────────────
ckpt = torch.load(
    os.path.join(CKPT_DIR, 'vjepa2_dinotxt_v2_epoch5.pt'),
    map_location=DEVICE
)
dinotxt.text_model.head.linear_projection.load_state_dict(ckpt)
print(f"Loaded epoch 5 checkpoint")
print(f"Projection norm: {dinotxt.text_model.head.linear_projection.weight.norm().item():.4f}")

# ── Confirm frozen state ───────────────────────────────────────────────────────
for name, param in dinotxt.named_parameters():
    param.requires_grad = 'text_model.head.linear_projection' in name
for param in _full_model.vjepa2.parameters():
    param.requires_grad = False
print(f"✓ Only linear_projection trainable")

# ── Optimizer with higher LR ───────────────────────────────────────────────────
NUM_EPOCHS           = 5
LR                   = 3e-4
WEIGHT_DECAY         = 0.0
GRADIENT_ACCUM_STEPS = 4

optimizer = torch.optim.AdamW(
    [p for p in dinotxt.parameters() if p.requires_grad],
    lr=LR,
    betas=(0.9, 0.99),
    eps=1e-8,
    weight_decay=WEIGHT_DECAY,
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

print(f"\nContinuing training — {NUM_EPOCHS} epochs, LR={LR}")
print(f"Starting from Val Top-1: 0.3807  Val Top-5: 0.6392\n")

# ── Evaluation ────────────────────────────────────────────────────────────────
def evaluate(loader):
    dinotxt.eval()
    with torch.no_grad():
        verb_embs = dinotxt.encode_text(verb_tokens_bare, normalize=True)
    correct_top1, correct_top5, total = 0, 0, 0
    with torch.no_grad():
        for video_embs, captions, labels in loader:
            video_embs = F.normalize(video_embs.to(DEVICE), dim=-1)
            labels     = labels.to(DEVICE)
            sim        = dinotxt.logit_scale.exp() * video_embs @ verb_embs.T
            correct_top1 += (sim.argmax(dim=-1) == labels).sum().item()
            top5          = sim.topk(5, dim=-1).indices
            correct_top5 += (top5 == labels.unsqueeze(-1)).any(dim=-1).sum().item()
            total        += labels.size(0)
    return correct_top1 / total, correct_top5 / total

# ── Training loop ─────────────────────────────────────────────────────────────
for epoch in range(1, NUM_EPOCHS + 1):
    dinotxt.train()
    optimizer.zero_grad()
    running_loss = 0.0

    print(f"Epoch {epoch}/{NUM_EPOCHS} — LR: {optimizer.param_groups[0]['lr']:.6f}")

    for step, (video_embs, captions, labels) in enumerate(
        tqdm(train_loader, desc="  Training"), start=1
    ):
        video_embs  = F.normalize(video_embs.to(DEVICE), dim=-1)
        labels      = labels.to(DEVICE)

        verb_embs   = dinotxt.encode_text(verb_tokens_bare, normalize=True)
        logit_scale = dinotxt.logit_scale.exp()
        sim         = logit_scale * video_embs @ verb_embs.T
        loss        = F.cross_entropy(sim, labels) / GRADIENT_ACCUM_STEPS
        loss.backward()
        running_loss += loss.item() * GRADIENT_ACCUM_STEPS

        if step % GRADIENT_ACCUM_STEPS == 0:
            optimizer.step()
            optimizer.zero_grad()
            with torch.no_grad():
                dinotxt.logit_scale.clamp_(0, math.log(100))

    if step % GRADIENT_ACCUM_STEPS != 0:
        optimizer.step()
        optimizer.zero_grad()
        with torch.no_grad():
            dinotxt.logit_scale.clamp_(0, math.log(100))

    scheduler.step()

    top1, top5   = evaluate(val_loader)
    avg_loss     = running_loss / len(train_loader)
    proj_norm    = dinotxt.text_model.head.linear_projection.weight.norm().item()
    print(f"  Loss: {avg_loss:.4f} | Val Top-1: {top1:.4f} | "
          f"Val Top-5: {top5:.4f} | logit_scale: {dinotxt.logit_scale.exp().item():.4f}")
    print(f"  Projection norm: {proj_norm:.4f}")

    ckpt_path = os.path.join(CKPT_DIR, f'vjepa2_dinotxt_v2_cont_epoch{epoch}.pt')
    torch.save(dinotxt.text_model.head.linear_projection.state_dict(), ckpt_path)
    print(f"  Checkpoint → {ckpt_path}")

print("\nTraining complete.")

Loaded epoch 5 checkpoint
Projection norm: 18.5302
✓ Only linear_projection trainable

Continuing training — 5 epochs, LR=0.0003
Starting from Val Top-1: 0.3807  Val Top-5: 0.6392

Epoch 1/5 — LR: 0.000300


  Training:   0%|          | 0/305 [00:00<?, ?it/s]

  Loss: 2.8148 | Val Top-1: 0.3834 | Val Top-5: 0.6426 | logit_scale: 14.2857
  Projection norm: 18.6693
  Checkpoint → /home/ubuntu/youcook2-videos/checkpoints/vjepa2_dinotxt_v2_cont_epoch1.pt
Epoch 2/5 — LR: 0.000271


  Training:   0%|          | 0/305 [00:00<?, ?it/s]

  Loss: 2.6903 | Val Top-1: 0.3779 | Val Top-5: 0.6405 | logit_scale: 14.2857
  Projection norm: 18.7412
  Checkpoint → /home/ubuntu/youcook2-videos/checkpoints/vjepa2_dinotxt_v2_cont_epoch2.pt
Epoch 3/5 — LR: 0.000196


  Training:   0%|          | 0/305 [00:00<?, ?it/s]

  Loss: 2.6389 | Val Top-1: 0.3810 | Val Top-5: 0.6558 | logit_scale: 14.2857
  Projection norm: 18.7712
  Checkpoint → /home/ubuntu/youcook2-videos/checkpoints/vjepa2_dinotxt_v2_cont_epoch3.pt
Epoch 4/5 — LR: 0.000104


  Training:   0%|          | 0/305 [00:00<?, ?it/s]

  Loss: 2.5906 | Val Top-1: 0.3886 | Val Top-5: 0.6466 | logit_scale: 14.2857
  Projection norm: 18.7667
  Checkpoint → /home/ubuntu/youcook2-videos/checkpoints/vjepa2_dinotxt_v2_cont_epoch4.pt
Epoch 5/5 — LR: 0.000029


  Training:   0%|          | 0/305 [00:00<?, ?it/s]

  Loss: 2.5610 | Val Top-1: 0.3865 | Val Top-5: 0.6530 | logit_scale: 14.2857
  Projection norm: 18.7610
  Checkpoint → /home/ubuntu/youcook2-videos/checkpoints/vjepa2_dinotxt_v2_cont_epoch5.pt

Training complete.


In [56]:
import torch.nn.functional as F

# ── Load best checkpoint ───────────────────────────────────────────────────────
ckpt = torch.load(
    os.path.join(CKPT_DIR, 'vjepa2_dinotxt_v2_cont_epoch5.pt'),
    map_location=DEVICE
)
dinotxt.text_model.head.linear_projection.load_state_dict(ckpt)
dinotxt.eval()
print("Loaded best checkpoint\n")

# ── Pick 3 videos from val set ─────────────────────────────────────────────────
sample_video_ids = df_val['video_id'].unique()[:3]
df_sample        = df_val[df_val['video_id'].isin(sample_video_ids)].reset_index(drop=True)

print(f"Showing results for {len(sample_video_ids)} videos ({len(df_sample)} segments)\n")

with torch.no_grad():
    for video_id in sample_video_ids:
        df_video = df_sample[df_sample['video_id'] == video_id].reset_index(drop=True)
        print(f"{'='*60}")
        print(f"Video: {video_id}  ({len(df_video)} segments)")
        print(f"{'='*60}")

        for _, row in df_video.iterrows():
            emb_path = os.path.join(EMB_DIR, f"{row['segment_id']}.pt")
            if not os.path.exists(emb_path):
                print(f"  MISSING: {row['segment_id']}")
                continue

            # video embedding
            video_emb = torch.load(emb_path, map_location=DEVICE)
            video_emb = F.normalize(video_emb.unsqueeze(0), dim=-1)

            # encode all 282 bare verbs
            CHUNK = 64
            all_embs = []
            for i in range(0, len(verb_vocab), CHUNK):
                tokens = tokenizer.tokenize(verb_vocab[i:i+CHUNK]).to(DEVICE)
                embs   = dinotxt.encode_text(tokens, normalize=True)
                all_embs.append(embs)
            verb_embs = torch.cat(all_embs, dim=0)  # (282, 1024)

            sim          = (dinotxt.logit_scale.exp() * video_emb @ verb_embs.T).squeeze(0)
            top5_indices = sim.topk(5).indices.tolist()
            top1_verb    = verb_vocab[top5_indices[0]]
            top5_verbs   = [verb_vocab[i] for i in top5_indices]
            gt_verb      = row['extracted_verb']
            correct      = top1_verb == gt_verb
            in_top5      = gt_verb in top5_verbs

            print(f"\n  Segment  : {row['segment_id']}")
            print(f"  Caption  : {row['masked_caption']}")
            print(f"  GT verb  : {gt_verb}")
            print(f"  Top-1    : {top1_verb}  {'✓' if correct else '✗'}")
            print(f"  Top-5    : {top5_verbs}  {'✓ in top-5' if in_top5 else '✗'}")
            print(f"  GT sim   : {sim[label2id[gt_verb]].item():.4f}")
            print(f"  Top1 sim : {sim[top5_indices[0]].item():.4f}")

        print()

Loaded best checkpoint

Showing results for 3 videos (21 segments)

Video: -AwyG1JcMp8  (5 segments)

  Segment  : -AwyG1JcMp8_000
  Caption  : [MASK] kimchi sausage soy sauce sesame oil green onion ginger and red pepper flakes
  GT verb  : combine
  Top-1    : add  ✗
  Top-5    : ['add', 'mix', 'cook', 'stir', 'pour']  ✗
  GT sim   : 0.9569
  Top1 sim : 4.6958

  Segment  : -AwyG1JcMp8_001
  Caption  : [MASK] the filling on a wrapper and press the edges together
  GT verb  : place
  Top-1    : cut  ✗
  Top-5    : ['cut', 'add', 'place', 'chop', 'slice']  ✓ in top-5
  GT sim   : 1.9176
  Top1 sim : 3.3509

  Segment  : -AwyG1JcMp8_002
  Caption  : [MASK] the dumplings above a pot of boiling water
  GT verb  : place
  Top-1    : add  ✗
  Top-5    : ['add', 'place', 'cook', 'boil', 'heat']  ✓ in top-5
  GT sim   : 2.1293
  Top1 sim : 2.6983

  Segment  : -AwyG1JcMp8_003
  Caption  : [MASK] soy sauce orange juice honey and sesame seeds
  GT verb  : mix
  Top-1    : add  ✗
  Top-5    : ['a

In [58]:
# Calculate Top-1 and Top-5 accuracy over 10 videos
sample_video_ids = df_val['video_id'].unique()[:10]
df_sample        = df_val[df_val['video_id'].isin(sample_video_ids)].reset_index(drop=True)

correct_top1 = 0
correct_top5 = 0
total        = 0

with torch.no_grad():
    for video_id in sample_video_ids:
        df_video = df_sample[df_sample['video_id'] == video_id].reset_index(drop=True)
        print(f"{'='*60}")
        print(f"Video: {video_id}  ({len(df_video)} segments)")
        print(f"{'='*60}")

        for _, row in df_video.iterrows():
            emb_path = os.path.join(EMB_DIR, f"{row['segment_id']}.pt")
            if not os.path.exists(emb_path):
                print(f"  MISSING: {row['segment_id']}")
                continue

            video_emb = torch.load(emb_path, map_location=DEVICE)
            video_emb = F.normalize(video_emb.unsqueeze(0), dim=-1)

            CHUNK = 64
            all_embs = []
            for i in range(0, len(verb_vocab), CHUNK):
                tokens = tokenizer.tokenize(verb_vocab[i:i+CHUNK]).to(DEVICE)
                embs   = dinotxt.encode_text(tokens, normalize=True)
                all_embs.append(embs)
            verb_embs = torch.cat(all_embs, dim=0)

            sim          = (dinotxt.logit_scale.exp() * video_emb @ verb_embs.T).squeeze(0)
            top5_indices = sim.topk(5).indices.tolist()
            top1_verb    = verb_vocab[top5_indices[0]]
            top5_verbs   = [verb_vocab[i] for i in top5_indices]
            gt_verb      = row['extracted_verb']
            top1_correct = top1_verb == gt_verb
            top5_correct = gt_verb in top5_verbs

            correct_top1 += int(top1_correct)
            correct_top5 += int(top5_correct)
            total        += 1

            print(f"\n  Segment  : {row['segment_id']}")
            print(f"  Caption  : {row['masked_caption']}")
            print(f"  GT verb  : {gt_verb}")
            print(f"  Top-1    : {top1_verb}  {'✓' if top1_correct else '✗'}")
            print(f"  Top-5    : {top5_verbs}  {'✓ in top-5' if top5_correct else '✗'}")
            print(f"  GT sim   : {sim[label2id[gt_verb]].item():.4f}")
            print(f"  Top1 sim : {sim[top5_indices[0]].item():.4f}")

        print()

# ── Summary ───────────────────────────────────────────────────────────────────
top1 = correct_top1 / total
top5 = correct_top5 / total

print(f"{'='*60}")
print(f"=== Summary ({len(sample_video_ids)} videos, {total} segments) ===")
print(f"{'='*60}")
print(f"  Top-1 accuracy        : {correct_top1}/{total} = {top1:.4f} ({top1*100:.2f}%)")
print(f"  Top-5 accuracy        : {correct_top5}/{total} = {top5:.4f} ({top5*100:.2f}%)")
print(f"  Random Top-1 baseline : {1/len(verb_vocab):.4f} ({100/len(verb_vocab):.2f}%)")
print(f"  Random Top-5 baseline : {5/len(verb_vocab):.4f} ({500/len(verb_vocab):.2f}%)")
print(f"  Top-1 vs random       : {top1/(1/len(verb_vocab)):.1f}x above chance")
print(f"  Top-5 vs random       : {top5/(5/len(verb_vocab)):.1f}x above chance")

Video: -AwyG1JcMp8  (5 segments)

  Segment  : -AwyG1JcMp8_000
  Caption  : [MASK] kimchi sausage soy sauce sesame oil green onion ginger and red pepper flakes
  GT verb  : combine
  Top-1    : add  ✗
  Top-5    : ['add', 'mix', 'cook', 'stir', 'pour']  ✗
  GT sim   : 0.9569
  Top1 sim : 4.6958

  Segment  : -AwyG1JcMp8_001
  Caption  : [MASK] the filling on a wrapper and press the edges together
  GT verb  : place
  Top-1    : cut  ✗
  Top-5    : ['cut', 'add', 'place', 'chop', 'slice']  ✓ in top-5
  GT sim   : 1.9176
  Top1 sim : 3.3509

  Segment  : -AwyG1JcMp8_002
  Caption  : [MASK] the dumplings above a pot of boiling water
  GT verb  : place
  Top-1    : add  ✗
  Top-5    : ['add', 'place', 'cook', 'boil', 'heat']  ✓ in top-5
  GT sim   : 2.1293
  Top1 sim : 2.6983

  Segment  : -AwyG1JcMp8_003
  Caption  : [MASK] soy sauce orange juice honey and sesame seeds
  GT verb  : mix
  Top-1    : add  ✗
  Top-5    : ['add', 'cook', 'pour', 'fry', 'mix']  ✓ in top-5
  GT sim   : 1.5462
 

In [ ]:
import json

with open('/home/ubuntu/youcook2-videos/processed/common_video_ids.json') as f:
    common = json.load(f)

common_segment_ids = set(common['segment_ids'])
print(f"Segments to evaluate: {len(common_segment_ids):,}")

In [46]:
import torch.nn.functional as F
from tqdm.auto import tqdm

# ── Load best checkpoint (epoch 2 had highest Top-1) ──────────────────────────
BEST_EPOCH = 2
ckpt_path  = os.path.join(CKPT_DIR, f'vjepa2_dinotxt_proj_epoch{BEST_EPOCH}.pt')
dinotxt.text_model.head.linear_projection.load_state_dict(
    torch.load(ckpt_path, map_location=DEVICE)
)
print(f"Loaded checkpoint: {ckpt_path}")

# ── Contextual evaluation ──────────────────────────────────────────────────────
# For each segment:
#   1. Fill each verb candidate into the masked caption
#   2. Encode all filled captions through DinoTXT text tower
#   3. Compare against the V-JEPA2 video embedding
#   4. Pick the verb whose filled caption is most similar to the video

dinotxt.eval()
results = []

with torch.no_grad():
    for _, row in tqdm(df_val.iterrows(), total=len(df_val), desc="Evaluating"):
        emb_path = os.path.join(EMB_DIR, f"{row['segment_id']}.pt")
        if not os.path.exists(emb_path):
            continue

        # video embedding — (1024,)
        video_emb = torch.load(emb_path, map_location=DEVICE)
        video_emb = F.normalize(video_emb.unsqueeze(0), dim=-1)  # (1, 1024)

        # fill each verb into the masked caption → (n_verbs,) sentences
        filled = [row['masked_caption'].replace('[MASK]', v) for v in verb_vocab]

        # encode in chunks to avoid OOM on large verb vocab
        CHUNK = 64
        all_text_embs = []
        for i in range(0, len(filled), CHUNK):
            chunk_tokens = tokenizer.tokenize(filled[i:i+CHUNK]).to(DEVICE)
            chunk_embs   = dinotxt.encode_text(chunk_tokens, normalize=True)
            all_text_embs.append(chunk_embs)
        text_embs = torch.cat(all_text_embs, dim=0)  # (n_verbs, 1024)

        # similarity
        logit_scale = dinotxt.logit_scale.exp()
        sim         = logit_scale * video_emb @ text_embs.T  # (1, n_verbs)
        sim         = sim.squeeze(0)                          # (n_verbs,)

        top5_indices = sim.topk(5).indices.tolist()
        top1_verb    = verb_vocab[top5_indices[0]]
        gt_verb      = row['extracted_verb']
        gt_idx       = label2id.get(gt_verb, -1)

        results.append({
            'segment_id'    : row['segment_id'],
            'masked_caption': row['masked_caption'],
            'gt_verb'       : gt_verb,
            'top1_pred'     : top1_verb,
            'top5_preds'    : [verb_vocab[i] for i in top5_indices],
            'top1_correct'  : top1_verb == gt_verb,
            'top5_correct'  : gt_idx in top5_indices,
            'filled_gt'     : row['masked_caption'].replace('[MASK]', gt_verb),
            'filled_top1'   : row['masked_caption'].replace('[MASK]', top1_verb),
        })

results_df = pd.DataFrame(results)

# ── Metrics ───────────────────────────────────────────────────────────────────
top1 = results_df['top1_correct'].mean()
top5 = results_df['top5_correct'].mean()

print(f"\n=== Contextual Evaluation (epoch {BEST_EPOCH}) ===")
print(f"  Evaluated segments : {len(results_df):,}")
print(f"  Val Top-1          : {top1:.4f}  ({top1*100:.2f}%)")
print(f"  Val Top-5          : {top5:.4f}  ({top5*100:.2f}%)")
print(f"  Random baseline    : {1/len(verb_vocab):.4f}  ({100/len(verb_vocab):.2f}%)")
print(f"  Top-1 vs random    : {top1/(1/len(verb_vocab)):.1f}x above chance")

# ── Per-class breakdown ───────────────────────────────────────────────────────
verb_counts    = df_train['extracted_verb'].value_counts()
frequent_verbs = set(verb_counts[verb_counts >= 50].index)
rare_verbs     = set(verb_counts[verb_counts < 10].index)

freq_mask = results_df['gt_verb'].isin(frequent_verbs)
rare_mask = results_df['gt_verb'].isin(rare_verbs)

print(f"\n=== Per-class breakdown ===")
print(f"  Frequent verbs (≥50 train) Top-1 : {results_df[freq_mask]['top1_correct'].mean():.4f}  ({freq_mask.sum()} samples)")
print(f"  Rare verbs     (<10 train) Top-1  : {results_df[rare_mask]['top1_correct'].mean():.4f}  ({rare_mask.sum()} samples)")

# ── Most confused verb pairs ───────────────────────────────────────────────────
wrong = results_df[~results_df['top1_correct']]
confusion = wrong.groupby(['gt_verb', 'top1_pred']).size().reset_index(name='count')
confusion = confusion.sort_values('count', ascending=False)

print(f"\n=== Top 10 confused verb pairs ===")
print(f"  {'Ground truth':<15} {'Predicted':<15} {'Count':>5}")
print(f"  {'-'*15} {'-'*15} {'-'*5}")
for _, row in confusion.head(10).iterrows():
    print(f"  {row['gt_verb']:<15} {row['top1_pred']:<15} {row['count']:>5}")

# ── Sample correct and incorrect predictions ───────────────────────────────────
print(f"\n=== Sample CORRECT predictions ===")
for _, row in results_df[results_df['top1_correct']].head(3).iterrows():
    print(f"  Caption : {row['masked_caption']}")
    print(f"  GT/Pred : {row['gt_verb']}")
    print()

print(f"=== Sample INCORRECT predictions ===")
for _, row in results_df[~results_df['top1_correct']].head(3).iterrows():
    print(f"  Caption : {row['masked_caption']}")
    print(f"  GT verb : {row['gt_verb']}")
    print(f"  Pred    : {row['top1_pred']}")
    print(f"  Top-5   : {row['top5_preds']}")
    print()

Loaded checkpoint: /home/ubuntu/youcook2-videos/checkpoints/vjepa2_dinotxt_proj_epoch2.pt


Evaluating:   0%|          | 0/3268 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [44]:
# Check what the masked captions look like vs bare verb labels
print("Sample masked caption:", df_val.iloc[0]['masked_caption'])
print("Sample verb label    :", df_val.iloc[0]['extracted_verb'])

Sample masked caption: [MASK] kimchi sausage soy sauce sesame oil green onion ginger and red pepper flakes
Sample verb label    : combine


In [ ]:
results_df = pd.DataFrame(history)
print("=== Training history ===")
print(results_df.to_string(index=False))

best = results_df.loc[results_df['val_top1'].idxmax()]
print(f"\nBest epoch    : {int(best['epoch'])}")
print(f"Best Val Top-1: {best['val_top1']:.4f}")
print(f"Best Val Top-5: {best['val_top5']:.4f}")

print("\n=== Model comparison (fill in from other notebooks) ===")
comparison = {
    'Model'         : ['DINOv2 + DinoTXT (NB4)', 'DINOv3 + DinoTXT (NB4)', 'V-JEPA2 video-only (NB5)', 'V-JEPA2 + DinoTXT (NB6 — this)'],
    'Val Top-1'     : ['???',                     '???',                     '???',                       f"{best['val_top1']:.4f}"],
    'Val Top-5'     : ['???',                     '???',                     '???',                       f"{best['val_top5']:.4f}"],
    'Projection?'   : ['No',                      'No',                      'No',                        'No'],
    'Text modality?': ['Yes',                      'Yes',                     'No',                        'Yes'],
}
print(pd.DataFrame(comparison).to_string(index=False))

## 12. Zero-shot evaluation on validation set

Same evaluation as NB4/NB5 for apples-to-apples comparison.

In [ ]:
text_encoder.eval()
eval_results = []
missing_embs = 0

with torch.no_grad():
    for _, row in tqdm(df_val.iterrows(), total=len(df_val), desc="Evaluating"):
        emb_path = os.path.join(EMB_DIR, f"{row['segment_id']}.pt")
        if not os.path.exists(emb_path):
            missing_embs += 1
            continue

        video_emb  = torch.load(emb_path, map_location=DEVICE)   # (1024,)
        video_emb  = video_emb / video_emb.norm()                 # L2 normalise

        # Replace with your NB4 text embedding call
        text_embs  = get_text_embeddings_for_segment(row['masked_caption'])  # (n_verbs, 2048)
        similarity = text_embs @ video_emb                         # (n_verbs,)

        top5_indices = similarity.topk(5).indices.tolist()
        top1_verb    = verb_vocab[top5_indices[0]]
        ground_truth = row['extracted_verb']
        gt_idx       = verb_vocab.index(ground_truth) if ground_truth in verb_vocab else -1

        eval_results.append({
            'segment_id'  : row['segment_id'],
            'ground_truth': ground_truth,
            'top1_pred'   : top1_verb,
            'top5_preds'  : [verb_vocab[i] for i in top5_indices],
            'top1_correct': top1_verb == ground_truth,
            'top5_correct': gt_idx in top5_indices,
            'masked_caption': row['masked_caption'],
        })

results_df_eval = pd.DataFrame(eval_results)
top1 = results_df_eval['top1_correct'].mean()
top5 = results_df_eval['top5_correct'].mean()

print(f"Predicted          : {len(eval_results):,}")
print(f"Missing embeddings : {missing_embs:,}")
print(f"\nVal Top-1 accuracy : {top1:.4f}")
print(f"Val Top-5 accuracy : {top5:.4f}")

# Per-class accuracy for frequent vs rare verbs
verb_counts  = df_train['extracted_verb'].value_counts()
frequent_verbs = set(verb_counts[verb_counts >= 50].index)
rare_verbs     = set(verb_counts[verb_counts < 10].index)

freq_mask = results_df_eval['ground_truth'].isin(frequent_verbs)
rare_mask = results_df_eval['ground_truth'].isin(rare_verbs)

print(f"\nFrequent verbs (≥50 train samples) Top-1 : {results_df_eval[freq_mask]['top1_correct'].mean():.4f}  ({freq_mask.sum()} samples)")
print(f"Rare verbs     (<10 train samples) Top-1  : {results_df_eval[rare_mask]['top1_correct'].mean():.4f}  ({rare_mask.sum()} samples)")